<a href="https://colab.research.google.com/github/marcocslima/rag_tests/blob/main/RAG_DEV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG com ChromaDB Local + BGE-M3 + Gemini

Sistema RAG completo com:
- **Embedding**: BAAI/bge-m3 (local, sem limites de API)
- **Vector Store**: ChromaDB persistente no Google Drive
- **LLM**: Gemini 2.5 Flash (via API)
- **OCR**: PyMuPDF + Tesseract para PDFs escaneados
- **Indexação incremental**: só processa PDFs novos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Instalação

In [ ]:
# Instalar dependências principais
%pip install -q -U google-genai
%pip install -q -U llama-index
%pip install -q -U llama-index-llms-google-genai
%pip install -q -U llama-index-embeddings-huggingface sentence-transformers
%pip install -q -U llama-index-vector-stores-chroma
%pip install -q -U chromadb

# Instalar dependências de OCR
!pip install -q pymupdf pytesseract
!apt-get install -y -q tesseract-ocr tesseract-ocr-por

## 2. Imports e Configuração

In [ ]:
import os
import io
import glob
import time
import shutil
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
from IPython.display import Markdown, display

from llama_index.core import (
    Document, Settings, StorageContext,
    VectorStoreIndex, PromptTemplate
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

import nest_asyncio
nest_asyncio.apply()

print("✅ Imports carregados")

## 3. API Key

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("✅ API Key configurada")

## 4. Configuração dos Modelos e ChromaDB

In [ ]:
def reset_database(confirm=False):
    """
    Apaga TODA a base vetorial e recria a collection vazia.
    Uso: reset_database(confirm=True)
    """
    global chroma_client, chroma_collection, vector_store, storage_context

    if not confirm:
        print("⚠️  Para confirmar o reset, chame: reset_database(confirm=True)")
        return

    print("🔴 Iniciando reset completo da base vetorial...")

    # 1. Fechar o client atual (liberar lock do SQLite)
    del chroma_client
    import gc
    gc.collect()
    print("   ✅ Client antigo liberado")

    # 2. Remover arquivos físicos do ChromaDB
    if os.path.exists(CHROMA_PATH):
        shutil.rmtree(CHROMA_PATH)
    os.makedirs(CHROMA_PATH, exist_ok=True)
    print("   ✅ Arquivos do ChromaDB removidos do Drive")

    # 3. Recriar tudo do zero (client novo, pasta limpa)
    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    print("   ✅ Collection recriada (vazia)")
    print(f"   📊 Chunks na base: {chroma_collection.count()}")
    print("\n🟢 Reset concluído! Rode a célula 6 para reindexar.")

In [ ]:
# Reset completo — apaga tudo e recria vazio
#reset_database(confirm=True)

In [ ]:
# =============================================
# CONFIGURAÇÕES - ajuste conforme necessário
# =============================================
PDF_FOLDER = "/content/drive/MyDrive/Tmp/RAG_DOCS"
CHROMA_PATH = "/content/drive/MyDrive/Tmp/RAG_DBS/CHROMA/chroma_db"
COLLECTION_NAME = "rag_collection"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 128

# =============================================
# Modelo de Embedding (local, sem limites)
# =============================================
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")

# =============================================
# LLM para geração de respostas
# =============================================
llm = GoogleGenAI(model_name="models/gemini-2.5-flash")

# =============================================
# ChromaDB persistente no Google Drive
# =============================================
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# =============================================
# Configurações globais do LlamaIndex
# =============================================
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = CHUNK_SIZE
Settings.chunk_overlap = CHUNK_OVERLAP

print("✅ Modelos e ChromaDB configurados")
print(f"   📊 Chunks na base: {chroma_collection.count()}")

## 5. Função de Extração de PDF (com OCR)

In [ ]:
def extract_pdf_text(pdf_path):
    """Extrai texto de PDF. Usa OCR automaticamente em páginas escaneadas."""
    doc = fitz.open(pdf_path)
    full_text = ""
    ocr_pages = 0

    for page_num, page in enumerate(doc):
        text = page.get_text()
        if len(text.strip()) < 50:  # página provavelmente é imagem → OCR
            pix = page.get_pixmap(dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(img, lang="por")
            ocr_pages += 1
        full_text += text + "\n"

    doc.close()

    if ocr_pages > 0:
        print(f"   🔍 OCR aplicado em {ocr_pages} página(s)")

    return full_text

print("✅ Função extract_pdf_text definida")

## 6. Indexação Incremental

Esta célula detecta automaticamente quais PDFs já foram indexados e processa apenas os novos.

In [ ]:
# =============================================
# Verificar quais PDFs já estão na base
# =============================================
existing_data = chroma_collection.get(include=["metadatas"])
indexed_sources = set()
for meta in existing_data["metadatas"]:
    if meta and "source" in meta:
        indexed_sources.add(meta["source"])

pdf_files = glob.glob(os.path.join(PDF_FOLDER, "*.pdf"))
new_pdfs = [f for f in pdf_files if f not in indexed_sources]

print(f"📁 PDFs na pasta: {len(pdf_files)}")
print(f"📊 Já indexados: {len(indexed_sources)}")
print(f"🆕 Novos para indexar: {len(new_pdfs)}")

# =============================================
# Extrair texto dos PDFs novos
# =============================================
all_documents = []
if new_pdfs:
    print()
    for pdf_path in new_pdfs:
        filename = os.path.basename(pdf_path)
        print(f"📄 Extraindo: {filename}")
        text = extract_pdf_text(pdf_path)
        doc = Document(text=text, metadata={"source": pdf_path, "filename": filename})
        all_documents.append(doc)
        print(f"   ✅ {len(text):,} caracteres extraídos")
else:
    print("\n✅ Nenhum PDF novo para processar.")

# =============================================
# Indexar documentos novos
# =============================================
if all_documents:
    print("\n🔄 Criando chunks e indexando...")
    splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    nodes = splitter.get_nodes_from_documents(all_documents)
    total = len(nodes)
    print(f"   📦 {total} chunks gerados")

    BATCH_SIZE = 10
    start = time.time()

    if chroma_collection.count() > 0:
        # Já existe índice → carregar e adicionar
        index = VectorStoreIndex.from_vector_store(vector_store)
        print("   📂 Índice existente carregado, adicionando novos chunks...\n")
        for i in range(0, total, BATCH_SIZE):
            batch = nodes[i:i + BATCH_SIZE]
            index.insert_nodes(batch)
            done = min(i + BATCH_SIZE, total)
            pct = (done / total) * 100
            elapsed = time.time() - start
            eta = (elapsed / done) * (total - done) if done > 0 else 0
            print(f"  ⏳ {done}/{total} chunks ({pct:.1f}%) — restante: ~{eta:.0f}s")
    else:
        # Criar do zero
        print("🔄 Criando índice do zero...\n")
        for i in range(0, total, BATCH_SIZE):
            batch = nodes[i:i + BATCH_SIZE]
            if i == 0:
                index = VectorStoreIndex(batch, storage_context=storage_context)
            else:
                index.insert_nodes(batch)
            done = min(i + BATCH_SIZE, total)
            pct = (done / total) * 100
            elapsed = time.time() - start
            eta = (elapsed / done) * (total - done) if done > 0 else 0
            print(f"  ⏳ {done}/{total} chunks ({pct:.1f}%) — restante: ~{eta:.0f}s")

    elapsed_total = time.time() - start
    print(f"\n✅ {total} novos chunks indexados em {elapsed_total:.1f}s")
    print(f"📊 Total na base: {chroma_collection.count()} chunks")
else:
    # Carregar índice existente
    index = VectorStoreIndex.from_vector_store(vector_store)
    print(f"📂 Índice carregado da base existente ({chroma_collection.count()} chunks)")

## 7. Configuração do Query Engine

In [ ]:
# =============================================
# PROMPT OTIMIZADO PARA LEGISLAÇÃO
# =============================================
template = """Você é um assistente jurídico especializado em legislação municipal.

INSTRUÇÕES IMPORTANTES:
1. Leia CUIDADOSAMENTE e COMPLETAMENTE todos os trechos de contexto abaixo.
2. O artigo ou informação procurada pode estar no MEIO de um trecho, não necessariamente no início.
3. Procure por números de artigos (Art.), parágrafos (§), incisos (I, II, III...) e alíneas (a, b, c...).
4. Transcreva o texto encontrado de forma COMPLETA, incluindo todos os parágrafos, incisos e alíneas.
5. Use APENAS o contexto fornecido para responder.
6. Se realmente não encontrar a informação em nenhum dos trechos, diga que não encontrou.

Contexto:
{context_str}

Pergunta: {query_str}

Resposta:"""

qa_prompt = PromptTemplate(template)

query_engine = index.as_query_engine(
    text_qa_template=qa_prompt,
    similarity_top_k=10,
)

print("✅ Query engine configurado (top_k=10)")

## 8. Consulta

In [ ]:
# =============================================
# FAÇA SUA PERGUNTA AQUI
# =============================================
pergunta = "Qual o texto do artigo 166 da Lei Complementar 460/2008?"

response = query_engine.query(pergunta)
display(Markdown(response.response))

---
## 🧪 Testes e Diagnóstico

Células auxiliares para debug. Não precisam ser executadas normalmente.

In [ ]:
# TESTE: Verificar conteúdo dos chunks retornados
pergunta_teste = "artigo 166 responsáveis retenção na fonte recolhimento ISS"

print("=== BUSCA DIRETA NO CHROMADB ===")
query_embedding = embed_model.get_query_embedding(pergunta_teste)
results = chroma_collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)
for i, doc in enumerate(results['documents'][0]):
    print(f"\n--- Resultado {i+1} (distância: {results['distances'][0][i]:.4f}) ---")
    print(doc[:500])

print("\n\n=== BUSCA VIA LLAMAINDEX ===")
retriever = index.as_retriever(similarity_top_k=5)
nodes_result = retriever.retrieve(pergunta_teste)
for i, node in enumerate(nodes_result):
    print(f"\n--- Node {i+1} (score: {node.score:.4f}) ---")
    print(node.text[:500])

In [ ]:
# TESTE: Verificar texto extraído dos PDFs
# (rode após a célula de extração, antes da indexação)
if all_documents:
    print(f"Total de documentos: {len(all_documents)}")
    print(f"\n--- Primeiros 1000 caracteres do documento 1 ---")
    print(all_documents[0].text[:1000])
else:
    print("Nenhum documento carregado. Rode a célula de extração primeiro.")

In [ ]:
# TESTE: Buscar texto específico nos chunks da base
busca = "Art. 166"
all_data = chroma_collection.get(include=["documents"])
encontrados = []
for i, doc in enumerate(all_data["documents"]):
    if busca.lower() in doc.lower():
        encontrados.append((i, doc))

print(f"🔍 '{busca}' encontrado em {len(encontrados)} chunk(s):\n")
for idx, (i, doc) in enumerate(encontrados):
    print(f"--- Chunk {i} ---")
    # Mostrar contexto ao redor da ocorrência
    pos = doc.lower().find(busca.lower())
    start = max(0, pos - 100)
    end = min(len(doc), pos + 500)
    print(f"...{doc[start:end]}...")
    print()

In [ ]:
# UTILITÁRIO: Listar todos os arquivos indexados
existing_data = chroma_collection.get(include=["metadatas"])
sources = set()
for meta in existing_data["metadatas"]:
    if meta and "source" in meta:
        sources.add(meta["source"])

print(f"📊 Total de chunks na base: {chroma_collection.count()}")
print(f"📁 Arquivos indexados ({len(sources)}):")
for s in sorted(sources):
    print(f"   • {os.path.basename(s)}")